# DDPM — barrido de pasos de muestreo (25 vs 50 vs 100) · Kaggle / T4

**Objetivo**: decidir cuántos pasos DDIM usar en la evaluación grande, comparando calidad y velocidad.
Para una submuestra estratificada por fase, genera con **100, 50 y 25 pasos** (mismo ruido inicial → comparación justa) y reporta:
- **Visual**: original vs 100 vs 50 vs 25, una fila por fase.
- **Cuantitativo**: R²(gen vs real) de las 4 métricas físicas por nº de pasos.
- **Velocidad**: ms por imagen de cada ajuste → factor de aceleración.

Solo DDPM. Inputs en `/kaggle/input` (no requiere kaggle.json).

In [ ]:
N = 600           # nº de θ (submuestra estratificada del test)
K = 8             # muestras por θ para el R² (basta para comparar pasos)
STEPS_LIST = [100, 50, 25]
THETA_CHUNK = 64  # θ por llamada (batch efectivo = THETA_CHUNK*K)
SEED = 42
GEN_SEED = 123
import os
print(f"N={N} K={K} | pasos: {STEPS_LIST}")

In [ ]:
import sys, time, math, importlib.util, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

def _find(pat):
    hits = glob.glob(pat, recursive=True)
    if not hits:
        raise FileNotFoundError(pat)
    return hits[0]

DATASET_PATH = _find('/kaggle/input/**/dataset_unificado_v2.npz')
DDPM_CKPT    = _find('/kaggle/input/**/ddpm_spines_final_39crop.pt')
METRICS_PATH = _find('/kaggle/input/**/metrics.py')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('dataset:', DATASET_PATH)
print('ddpm:', DDPM_CKPT)
print('metrics:', METRICS_PATH)

spec = importlib.util.spec_from_file_location('metrics', METRICS_PATH)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules['metrics'] = metrics
from metrics import (STRUCTURE_NAMES, STRUCTURE_COLORS, MASK, IMG_SIZE,
                     get_structure_label, physical_metrics,
                     PHYSICAL_METRIC_NAMES, PHYSICAL_METRIC_LABELS,
                     apply_figure_style)
apply_figure_style()
print('fases:', STRUCTURE_NAMES)

In [ ]:
# Split de test (SEED=42) + scaler + submuestra estratificada por fase
data   = np.load(DATASET_PATH)
imgs   = data['img'].astype(np.float32)
params = data['params'].astype(np.float32)
labels = data['labels'].astype(int)
if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]
Nfull = len(imgs)

rng = np.random.RandomState(SEED)
sub_idx = rng.choice(Nfull, size=Nfull, replace=False)
idx_all = np.arange(len(sub_idx))
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=SEED)
idx_va, idx_te  = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)
scaler = MinMaxScaler().fit(params[sub_idx][idx_tr])

test_global = sub_idx[idx_te]
test_imgs   = imgs[test_global]
test_params = params[test_global]
test_phase  = np.array([get_structure_label(c) for c in labels[test_global]])

srng = np.random.RandomState(GEN_SEED)
sel = []
for ph in STRUCTURE_NAMES:
    idx_ph = np.where(test_phase == ph)[0]
    if len(idx_ph) == 0:
        continue
    q = min(int(round(N * len(idx_ph) / len(test_phase))), len(idx_ph))
    if q > 0:
        sel.append(srng.choice(idx_ph, size=q, replace=False))
sel = np.concatenate(sel)
srng.shuffle(sel)

eval_imgs  = test_imgs[sel]
eval_phase = test_phase[sel]
eval_cond  = scaler.transform(test_params[sel]).astype(np.float32)
n_theta = len(sel)
print(f'submuestra: {n_theta} theta')
for ph in STRUCTURE_NAMES:
    print(f'  {ph:24s} {int((eval_phase==ph).sum()):4d}')

In [ ]:
# Physical metrics on GPU (differentiable-free replica of metrics.py) + reference values.
# Canonical set is the three of metrics.py: magnetization, spin_correlation, peak_wave_vector.
METRICS = list(PHYSICAL_METRIC_NAMES)
METRIC_LABEL = {'magnetization': 'Magnetization',
                'spin_correlation': 'Spin Correlation',
                'peak_wave_vector': 'Peak Wave Vector'}

_H = _W = IMG_SIZE
_MASK_T = torch.tensor(MASK, dtype=torch.bool, device=DEVICE)
_MASK_F = _MASK_T.to(torch.float32)
_cy, _cx = _H // 2, _W // 2
_Y, _X = np.ogrid[:_H, :_W]
_R = np.sqrt((_X - _cx) ** 2 + (_Y - _cy) ** 2).astype(int)
_max_r = min(_cy, _cx)
_R_T = torch.tensor(_R, dtype=torch.long, device=DEVICE)
_RING_IDX = [(_R_T == r) for r in range(1, _max_r + 1)]
_RING_R = torch.tensor(list(range(1, _max_r + 1)), dtype=torch.float32, device=DEVICE)
_NN = []
for _dy, _dx in [(0, 1), (1, 0)]:
    _valid = MASK[:-_dy or None, :-_dx or None] & MASK[_dy:, _dx:]
    _NN.append((_dy, _dx, torch.tensor(_valid, dtype=torch.bool, device=DEVICE)))


def three_metrics_gpu(x):
    """(B, 39, 39) in [-1,1] -> (B, 3) in PHYSICAL_METRIC_NAMES order.

    Mirrors metrics.py exactly, including the fluctuation-field structure factor
    (disk mean removed, background zeroed) and the rad/site units of q_peak.
    Input must already be cropped to 39x39.
    """
    assert x.shape[-2:] == (_H, _W), f"expected {_H}x{_W}, got {tuple(x.shape[-2:])}"
    x = x.to(DEVICE, dtype=torch.float32)

    mag = x[:, _MASK_T].mean(dim=1)

    total = torch.zeros(x.shape[0], device=DEVICE)
    count = 0
    for dy, dx, vm in _NN:
        a = x[:, :x.shape[1] - dy if dy else None, :x.shape[2] - dx if dx else None]
        b = x[:, dy:, dx:]
        total = total + (a * b)[:, vm].sum(dim=1)
        count += int(vm.sum())
    corr = total / count

    delta = (x - mag[:, None, None]) * _MASK_F          # fluctuation field
    ft = torch.fft.fftshift(torch.fft.fft2(delta), dim=(-2, -1))
    sq = (ft.abs() ** 2) / (_H * _W)
    rm = torch.stack([sq[:, ring].mean(dim=1) for ring in _RING_IDX], dim=1)
    qpeak = 2.0 * math.pi * _RING_R[rm.argmax(dim=1)] / float(_H)
    qpeak = torch.where(rm.amax(dim=1) > 0, qpeak, torch.full_like(qpeak, float('nan')))

    return torch.stack([mag, corr, qpeak], dim=1).cpu().numpy()


# Parity check against metrics.py before trusting the GPU replica.
_chk = torch.from_numpy(eval_imgs[:8, :, :, 0].astype(np.float32))
_gpu = three_metrics_gpu(_chk)
_cpu = np.array([[physical_metrics(eval_imgs[i, :, :, 0])[m] for m in METRICS]
                 for i in range(8)])
_diff = np.nanmax(np.abs(_gpu - _cpu), axis=0)
print('max |GPU - metrics.py|:', dict(zip(METRICS, _diff.round(7))))
assert np.all(_diff < 1e-5), 'GPU replica does not match metrics.py'

real_arr = np.array([[physical_metrics(eval_imgs[i, :, :, 0])[m] for m in METRICS]
                     for i in range(n_theta)])
real_vals = {m: real_arr[:, j] for j, m in enumerate(METRICS)}
print('reference metrics OK')


In [ ]:
# Arquitectura DDPM (idéntica a ddpm_spines_train) + carga checkpoint
IMG_SIZE, CROP_TO, COND_DIM, T_STEPS = 40, 39, 8, 1000

class DDPMScheduler:
    def __init__(self, T=1000, schedule='cosine', device='cpu'):
        self.T = T
        steps = T+1; s = 0.008
        x = torch.linspace(0, T, steps, device=device)
        ac = torch.cos(((x/T)+s)/(1+s)*math.pi*0.5)**2
        ac = ac/ac[0]
        betas = (1-(ac[1:]/ac[:-1])).clamp(max=0.999)
        alphas = 1.0-betas
        ac2 = torch.cumprod(alphas, 0)
        self.sqrt_alphas_cumprod = ac2.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0-ac2).sqrt()

def sinusoidal_embedding(t, dim):
    half = dim//2
    freqs = torch.exp(-math.log(10000)*torch.arange(half, device=t.device).float()/(half-1))
    args = t[:,None].float()*freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in,out_dim), nn.SiLU(), nn.Linear(out_dim,out_dim))
    def forward(self, t, cond):
        return self.t_mlp(sinusoidal_embedding(t, self.t_mlp[0].in_features)) + self.c_mlp(cond)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch); self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch); self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, emb):
        h = F.silu(self.norm1(x)); h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h)); h = self.dropout(h); h = self.conv2(h)
        return h + self.skip(x)

class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch); self.qkv = nn.Conv2d(ch, ch*3, 1); self.proj = nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x); q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B, C, -1); k = k.reshape(B, C, -1); v = v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)

class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1,2,4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch*m for m in ch_mults]
        self.emb = TimeCondEmbedding(emb_dim, cond_dim, emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)
        self.down_blocks = nn.ModuleList(); self.down_samples = nn.ModuleList()
        in_ch = chs[0]; self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([ResBlock(in_ch,out_ch,emb_dim,dropout=dropout), ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.mid_block1 = ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.mid_attn = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1],chs[-1],emb_dim,dropout=dropout)
        self.up_blocks = nn.ModuleList(); self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i+1)]
            self.up_blocks.append(nn.ModuleList([ResBlock(in_ch+skip_ch,out_ch,emb_dim,dropout=dropout), ResBlock(out_ch,out_ch,emb_dim,dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch,out_ch,4,stride=2,padding=1) if i<len(chs)-1 else nn.Identity())
            in_ch = out_ch
        self.norm_out = nn.GroupNorm(8, chs[0]); self.conv_out = nn.Conv2d(chs[0], img_channels, 1)
    def forward(self, x, t, cond):
        emb = self.emb(t, cond); h = self.conv_in(x); skips = []
        for (r1,r2), ds in zip(self.down_blocks, self.down_samples):
            h = r1(h,emb); h = r2(h,emb); skips.append(h); h = ds(h)
        h = self.mid_block1(h,emb); h = self.mid_attn(h); h = self.mid_block2(h,emb)
        for (r1,r2), us, sk in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h,sk],1); h = r1(h,emb); h = r2(h,emb); h = us(h)
        return self.conv_out(F.silu(self.norm_out(h)))

@torch.no_grad()
def fast_sample(model, cond, sch, n_steps, img_size=40):
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    ts = list(range(0, sch.T, sch.T//n_steps))[::-1]
    for tv in ts:
        tt = torch.full((B,), tv, device=cond.device, dtype=torch.long)
        eps = model(x, tt, cond)
        sa = sch.sqrt_alphas_cumprod[tv]; s1 = sch.sqrt_one_minus_alphas_cumprod[tv]
        x0 = ((x - s1*eps)/sa).clamp(-1, 1)
        if tv > 0:
            tp = max(tv - sch.T//n_steps, 0)
            x = sch.sqrt_alphas_cumprod[tp]*x0 + sch.sqrt_one_minus_alphas_cumprod[tp]*eps
        else:
            x = x0
    return x

ckpt = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
hp = ckpt.get('hyperparams', {'base_ch':80,'cond_emb_dim':128,'dropout':0.1,'beta_schedule':'cosine'})
model = ConditionalUNet(base_ch=hp['base_ch'], emb_dim=hp['cond_emb_dim'], dropout=hp['dropout']).to(DEVICE)
model.load_state_dict(ckpt['model'])
if ckpt.get('ema') is not None:
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad and n in ckpt['ema']:
                p.data.copy_(ckpt['ema'][n].to(DEVICE))
model.eval()
sch = DDPMScheduler(T=T_STEPS, schedule=hp['beta_schedule'], device=DEVICE)
print(f"DDPM cargado: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Barrido: generar con cada nº de pasos, medir R² y velocidad.
# Las K muestras de cada θ se fusionan en el batch (θ × K).
def r2_of(g, r, mask=None):
    gg = g if mask is None else g[mask]
    rr = r if mask is None else r[mask]
    ok = ~(np.isnan(gg) | np.isnan(rr))
    if ok.sum() < 2 or np.var(rr[ok]) < 1e-12:
        return np.nan
    return float(r2_score(rr[ok], gg[ok]))

results = {}
for steps in STEPS_LIST:
    torch.manual_seed(GEN_SEED)
    gen_vals = {m: np.full(n_theta, np.nan) for m in METRICS}
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time(); n_imgs = 0
    for i0 in range(0, n_theta, THETA_CHUNK):
        i1 = min(i0+THETA_CHUNK, n_theta); chunk = i1-i0
        cond_rep = np.repeat(eval_cond[i0:i1], K, axis=0)
        c = torch.from_numpy(cond_rep).to(DEVICE)
        g40 = fast_sample(model, c, sch, steps, IMG_SIZE)
        g39 = g40[:, 0, :CROP_TO, :CROP_TO]
        met = three_metrics_gpu(g39).reshape(chunk, K, len(METRICS))
        mk = np.nanmean(met, axis=1)
        for jm, m in enumerate(METRICS):
            gen_vals[m][i0:i1] = mk[:, jm]
        n_imgs += chunk*K
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    dt = time.time()-t0
    results[steps] = {'metrics': gen_vals, 'sec_per_img': dt/n_imgs, 'total_s': dt}
    print(f'pasos={steps:3d} | {dt:6.1f}s | {1000*dt/n_imgs:.2f} ms/img')

print('\nR2 GENERAL por numero de pasos:')
header = f"{'pasos':>6} " + ' '.join(f'{METRIC_LABEL[m][:12]:>13}' for m in METRICS) + f"{'ms/img':>10}"
print(header)
for steps in STEPS_LIST:
    r2s = [r2_of(results[steps]['metrics'][m], real_vals[m]) for m in METRICS]
    row = f'{steps:>6} ' + ' '.join(f'{v:>13.3f}' for v in r2s) + f"{1000*results[steps]['sec_per_img']:>10.2f}"
    print(row)

In [ ]:
# Gráfica cuantitativa: R² por métrica y velocidad relativa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(METRICS)); w = 0.8/len(STEPS_LIST)
for i, steps in enumerate(STEPS_LIST):
    r2s = [r2_of(results[steps]['metrics'][m], real_vals[m]) for m in METRICS]
    axes[0].bar(x + (i-(len(STEPS_LIST)-1)/2)*w, r2s, w, label=f'{steps} steps')
axes[0].axhline(0, color='k', lw=.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([METRIC_LABEL[m] for m in METRICS], rotation=20, ha='right')
axes[0].set_ylabel('R2 (generated vs real)')
axes[0].set_title('Quality vs sampling steps')
axes[0].legend(); axes[0].grid(alpha=.3, axis='y')

base = results[STEPS_LIST[0]]['sec_per_img']
speed = [base/results[s]['sec_per_img'] for s in STEPS_LIST]
axes[1].bar([str(s) for s in STEPS_LIST], speed, color='#2563EB')
axes[1].set_xlabel('sampling steps')
axes[1].set_ylabel(f'speed-up vs {STEPS_LIST[0]} steps')
axes[1].set_title('Relative speed')
axes[1].grid(alpha=.3, axis='y')
for i, s in enumerate(speed):
    axes[1].text(i, s, f'{s:.1f}x', ha='center', va='bottom')
plt.tight_layout(); plt.savefig('ddpm_steps_quality_speed.png', dpi=130); plt.show()

In [ ]:
# Comparación VISUAL: original vs 100 vs 50 vs 25, una fila por fase.
# Mismo θ y mismo ruido inicial en los 3 ajustes → la diferencia es solo los pasos.
phases = [p for p in STRUCTURE_NAMES if (eval_phase==p).any()]
ncol = 1 + len(STEPS_LIST)
fig, axes = plt.subplots(len(phases), ncol, figsize=(2.6*ncol, 2.6*len(phases)))
vrng = np.random.RandomState(GEN_SEED)
for r, ph in enumerate(phases):
    pick = vrng.choice(np.where(eval_phase==ph)[0])
    cond1 = torch.from_numpy(eval_cond[pick:pick+1]).to(DEVICE)
    axes[r,0].imshow(eval_imgs[pick,:,:,0], cmap='jet', vmin=-1, vmax=1)
    axes[r,0].set_ylabel(ph, fontsize=8)
    if r == 0:
        axes[r,0].set_title('Original', fontsize=9)
    for c, steps in enumerate(STEPS_LIST):
        torch.manual_seed(GEN_SEED + int(pick))
        g = fast_sample(model, cond1, sch, steps, IMG_SIZE)[0,0,:CROP_TO,:CROP_TO].cpu().numpy()
        axes[r,c+1].imshow(g, cmap='jet', vmin=-1, vmax=1)
        if r == 0:
            axes[r,c+1].set_title(f'{steps} steps', fontsize=9)
    for c in range(ncol):
        axes[r,c].set_xticks([]); axes[r,c].set_yticks([])
fig.suptitle('DDPM - sampling steps: original vs 100 / 50 / 25', fontweight='bold')
plt.tight_layout(); plt.savefig('ddpm_steps_visual.png', dpi=130); plt.show()
print('Listo. Compara R2, ms/img y la calidad visual para elegir el numero de pasos.')